# Session 06 · MongoDB Shell Methods

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Setup

In [2]:
from campus_data import seed
from mongo_conn import get_db
db = get_db("campus_shell")
for c in db.list_collection_names():
    db[c].drop()
print(seed(db))

{'students': 10, 'batches': 3, 'payments': 12}


## Example 1 - sort, skip, limit

In [3]:
page1 = [(d["name"], d["marks"]["python"]) for d in
         db.students.find({}, {"_id": 0, "name": 1, "marks.python": 1}).sort("marks.python", -1).limit(3)]
page2 = [d["name"] for d in db.students.find({}, {"_id": 0, "name": 1}).sort("marks.python", -1).skip(3).limit(3)]
print(page1, page2, sep="\n")
assert page1[0][0] == "Faraz Iqbal"

[('Faraz Iqbal', 95), ('Sana Khan', 92), ('Zoya Hussain', 88)]
['Ayesha Noor', 'Ali Raza', 'Hina Shah']


## Example 2 - multi-key sort

In [4]:
for d in db.students.find({}, {"_id": 0, "batch": 1, "name": 1, "marks.python": 1}).sort([("batch", 1), ("marks.python", -1)]):
    print(d["batch"], d["marks"]["python"], d["name"])

AI-01 88 Zoya Hussain
AI-01 73 Hina Shah
AI-01 67 Bilal Ahmed
AI-01 60 Mariam Javed
DS-01 92 Sana Khan
DS-01 78 Ali Raza
DS-01 45 Usman Tariq
DS-02 95 Faraz Iqbal
DS-02 81 Ayesha Noor
DS-02 55 Hamza Ali


## Example 3 - iterating a cursor manually

In [5]:
cur = db.students.find({"city": "Karachi"}).sort("student_id", 1)
first, second = next(cur), next(cur)
rest = [d["name"] for d in cur]
print(first["name"], "|", second["name"], "|", rest)

Ali Raza | Hamza Ali | ['Bilal Ahmed', 'Usman Tariq', 'Faraz Iqbal']


## Example 4 - atomic find_one_and_update

In [6]:
from pymongo import ReturnDocument
for _ in range(3):
    counter = db.counters.find_one_and_update({"_id": "roll_no"}, {"$inc": {"seq": 1}},
                                              upsert=True, return_document=ReturnDocument.AFTER)
print("next roll number:", counter["seq"])
assert counter["seq"] == 3

next roll number: 3


## Example 5 - replace_one vs update_one

In [7]:
db.batches.replace_one({"_id": "DS-02"}, {"programme": "Data Science", "room": 9})
db.batches.update_one({"_id": "AI-01"}, {"$set": {"room": 7}})
print(db.batches.find_one({"_id": "DS-02"}))
print(sorted(db.batches.find_one({"_id": "AI-01"}).keys()))

{'_id': 'DS-02', 'programme': 'Data Science', 'room': 9}
['_id', 'capacity', 'faculty', 'programme', 'room', 'start']


## Example 6 - exact vs estimated counts, sibling database

In [8]:
print(db.students.count_documents({"batch": "DS-01"}), db.students.estimated_document_count())
shop = db.client["shop_demo"]
print("sibling db name:", shop.name)

3 10
sibling db name: shop_demo
